In [1]:
import torch
from torch import nn

In [2]:
class LogisticNet(nn.Module):
    def __init__(self):
        super(LogisticNet, self).__init__()
        self.pol1 = nn.MaxPool3d(2)
        self.fc1 = nn.Linear(2048, 4)
        self.act1 = nn.Softmax(dim=1)

    def forward(self, input):
        out = torch.flatten(self.pol1(input), 1)
        out = self.fc1(out)
        #out = self.act1(out)
        return out


class ReducedNet(nn.Module):
    def __init__(self):
        super(ReducedNet, self).__init__()
        self.cv1 = nn.Conv3d(1, 1, kernel_size=(5, 3, 5), stride=(2, 1, 2), padding=0)
        self.pol1 = nn.MaxPool3d(2)
        self.fc1 = nn.Linear(343, 128)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 4)
        self.act2 = nn.Softmax(dim=1)

    def forward(self, input):
        out = self.cv1(input)
        out = self.pol1(out)
        out = self.fc1(torch.flatten(out, 1))
        out = self.act1(out)
        out = self.fc2(out)
        #out = self.act2(out)
        return out


class BaselineNet(nn.Module):
    def __init__(self):
        super(BaselineNet, self).__init__()
        self.cv1 = nn.Conv3d(1, 32, kernel_size=(5, 3, 5), stride=(2, 1, 2), padding=0)
        self.cv2 = nn.Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=0)
        self.pol1 = nn.MaxPool3d(2)
        self.fc1 = nn.Linear(6912, 128)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 4)
        self.act2 = nn.Softmax(dim=1)

    def forward(self, input):
        out = self.cv1(input)
        out = self.cv2(out)
        out = self.pol1(out)
        out = self.fc1(torch.flatten(out, 1))
        out = self.act1(out)
        out = self.fc2(out)
        #out = self.act2(out)
        return out

model1 = LogisticNet()
model2 = ReducedNet()
model3 = BaselineNet()

In [3]:
model1.load_state_dict(torch.load('PreTrained_Weights/LogisticNet_84.pth'))
model2.load_state_dict(torch.load('PreTrained_Weights/ReducedNet_42.pth'))
model3.load_state_dict(torch.load('PreTrained_Weights/BaselineNet_336.pth'))

<All keys matched successfully>

In [4]:

tensor_x = torch.rand((1, 1, 32, 16, 32), dtype=torch.float32)
for model in [model1, model2, model3]:
    torch.onnx.export(
        model,
        (tensor_x,),
        f"{model.__class__.__name__}.onnx",  # filename of the ONNX model
        input_names=["input"],  # Rename inputs for the ONNX model
        output_names=["output"],  # Rename outputs for the ONNX model
        dynamo=False,  # True or False to select the exporter to use
    )

In [5]:
!../onnx2c/build/onnx2c LogisticNet.onnx > C/LogisticNet.c
!../onnx2c/build/onnx2c ReducedNet.onnx > C/ReducedNet.c
!../onnx2c/build/onnx2c BaselineNet.onnx > C/BaselineNet.c